# 阶段二实操：从深度图生成物体点云，并与 CAD 对照

## 阶段目标

阶段一使用数据集提供的真值位姿，将 CAD 变换到相机坐标系。本阶段加入深度图：使用实例可见掩码筛选深度像素并反投影为观测点云，再将 CAD 点云变换到相机坐标系进行比较。

| 已知输入 | 处理流程 | 结果 |
|---|---|---|
| 深度图、实例可见掩码、相机内参 K、depth_scale、CAD、真值位姿 T_CO | 深度反投影；CAD 位姿变换；坐标系统一 | 同一相机坐标系中的可见观测点云和完整 CAD 点云 |

**本阶段不进行位姿估计。**练习使用数据集提供的 `mask_visib` 和真值 `T_CO`，用于分析 RGB-D 点云与 CAD 几何的关系。`mask_visib` 是标注掩码，不是神经网络预测的分割结果；CAD 点云表示完整表面，深度点云通常仅覆盖可见表面。

## 数据与样本

Notebook 使用项目中的 YCB-V BOP 数据：

~~~text
.local/data/bop/ycbv/
├── models/
│   ├── models_info.json
│   └── obj_XXXXXX.ply
└── test/
    └── <scene_id>/
        ├── depth/图像编号.png
        ├── mask_visib/图像编号_gt_id.png
        ├── rgb/图像编号.png
        ├── scene_camera.json
        ├── scene_gt.json
        └── scene_gt_info.json
~~~

本 Notebook 与阶段一使用相同的样本选择规则：默认扫描 `test` 下的全部场景，选择 `visib_fract` 最高的实例。场景范围可通过准备单元中的 `SCENE_ID` 设置；`None` 表示扫描全部场景。执行后会打印实际选中的场景、图像编号、`gt_id` 和 `obj_id`。

## 准备：导入工具并设置数据路径

本单元导入点云、图像可视化和数值计算所需的库，并设置数据集目录。

In [ ]:
import json
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "6D位姿估计":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / ".local" / "data" / "bop" / "ycbv"
SPLIT_DIR = DATASET_ROOT / "test"
MODELS_DIR = DATASET_ROOT / "models"

# None 表示扫描 test 下的全部场景；整数表示只扫描指定场景。
SCENE_ID = None


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


## 1. 选择场景、图像和实例

`scene_gt.json` 为每张图像保存实例列表；`scene_gt_info.json` 中相同列表位置记录该实例的 `visib_fract` 等信息。代码与阶段一采用相同规则：默认扫描 `test` 下的全部场景，并选择可见比例最高的实例。

场景范围可通过准备单元中的 `SCENE_ID` 设置；`None` 表示扫描全部场景。代码读取数据集预先保存的可见比例，不在本单元重新计算。

In [ ]:
scene_dirs = sorted(
    path for path in SPLIT_DIR.iterdir()
    if path.is_dir() and path.name.isdigit()
)
if SCENE_ID is not None:
    scene_dirs = [path for path in scene_dirs if path.name == f"{SCENE_ID:06d}"]
if not scene_dirs:
    raise FileNotFoundError(f"在 {SPLIT_DIR} 中没有找到符合条件的场景目录。")

best = None
for candidate_scene_dir in scene_dirs:
    candidate_gt = read_json(candidate_scene_dir / "scene_gt.json")
    candidate_info = read_json(candidate_scene_dir / "scene_gt_info.json")

    for image_key, instances in candidate_gt.items():
        for candidate_gt_id, instance in enumerate(instances):
            info = candidate_info[image_key][candidate_gt_id]
            visibility = float(info.get("visib_fract", 0.0))
            if best is None or visibility > best[0]:
                best = (
                    visibility,
                    candidate_scene_dir,
                    int(image_key),
                    candidate_gt_id,
                    instance,
                    info,
                )

if best is None:
    raise ValueError("所选场景范围内没有可用的实例标注。")

visibility, scene_dir, image_id, gt_id, gt, gt_info = best
scene_gt = read_json(scene_dir / "scene_gt.json")
scene_camera = read_json(scene_dir / "scene_camera.json")
camera = scene_camera[str(image_id)]
obj_id = int(gt["obj_id"])

print(f"场景编号: {scene_dir.name}")
print(f"图像编号: {image_id:06d}")
print(f"实例序号 gt_id: {gt_id}")
print(f"物体模型编号 obj_id: {obj_id}")
print(f"该实例的可见比例: {visibility:.3f}")

## 2. 读取深度、掩码和相机参数

`mask_visib` 是按实例区分的二值掩码：非零像素标记该实例在图像中实际可见的区域，被其他物体遮挡的部分不包含在内。BOP 的 `mask` 表示物体轮廓，`mask_visib` 只表示可见部分。

掩码文件名为 `<image_id>_<gt_id>.png`。其中 `gt_id` 是当前图像在 `scene_gt.json` 实例列表中的序号，从 0 开始；它用于将掩码对应到具体实例。掩码来自数据集标注，不是算法预测结果。

本单元同时使用可见掩码和有效深度条件，因此只保留该实例可见且深度有效的像素。深度图中的 0 表示无有效深度，此类像素会被排除。

BOP 的 `depth_scale` 用于将深度图原始数值换算为毫米：

$$
Z_C = D_{\mathrm{raw}} \times \mathrm{depth\_scale}
$$

其中 $Z_C$ 是相机坐标系中的深度。

In [ ]:
depth_path = scene_dir / "depth" / f"{image_id:06d}.png"
mask_path = scene_dir / "mask_visib" / f"{image_id:06d}_{gt_id:06d}.png"
rgb_path = scene_dir / "rgb" / f"{image_id:06d}.png"

# 读取 BOP 深度图、当前实例的可见掩码和 RGB 图像。
depth_raw = np.asarray(o3d.io.read_image(str(depth_path)))
mask_raw = np.asarray(o3d.io.read_image(str(mask_path)))
rgb = np.asarray(o3d.io.read_image(str(rgb_path)))[:, :, :3]

# 某些图像读取器会为单通道图像保留一个长度为 1 的通道维。
if depth_raw.ndim == 3:
    depth_raw = depth_raw[:, :, 0]
if mask_raw.ndim == 3:
    mask_raw = mask_raw[:, :, 0]

# 掩码中非零像素表示当前实例的可见区域。
visible_mask = mask_raw > 0

# depth_scale 来自同一图像的 scene_camera.json；转换结果以毫米为单位。
depth_scale = float(camera["depth_scale"])
depth_mm = depth_raw.astype(np.float32) * depth_scale

# 深度值为 0 或非有限值时，不将该像素反投影为三维点。
valid_depth_mask = np.isfinite(depth_mm) & (depth_mm > 0) & (depth_raw > 0)
object_depth_mask = visible_mask & valid_depth_mask

if depth_raw.shape != visible_mask.shape or rgb.shape[:2] != visible_mask.shape:
    raise ValueError("RGB、深度图和实例掩码的图像尺寸不一致。")
if not np.any(object_depth_mask):
    raise ValueError("当前实例没有同时具备可见掩码和有效深度的像素。")

K = np.asarray(camera["cam_K"], dtype=float).reshape(3, 3)
R_co = np.asarray(gt["cam_R_m2c"], dtype=float).reshape(3, 3)
t_co = np.asarray(gt["cam_t_m2c"], dtype=float).reshape(3)

print(f"深度文件: {depth_path}")
print(f"可见掩码: {mask_path}")
print(f"depth_scale: {depth_scale}")
print(f"可见掩码像素数: {np.count_nonzero(visible_mask)}")
print(f"具有有效深度的目标像素数: {np.count_nonzero(object_depth_mask)}")
print(f"目标有效深度范围 (mm): {depth_mm[object_depth_mask].min():.1f} – {depth_mm[object_depth_mask].max():.1f}")

### 输入图像、掩码与深度显示

左图以绿色半透明叠加当前实例的 `mask_visib` 区域；绿色只是掩码的可视化标记，不是物体本身的颜色。右图显示该掩码内具有有效深度的像素，颜色表示深度大小。该可视化用于检查实例区域与有效深度筛选结果。

In [ ]:
# 将可见掩码叠加到 RGB 图像，便于核对当前实例区域。
rgb_overlay = rgb.copy()
green = np.array([0, 255, 0], dtype=np.float32)
rgb_overlay[visible_mask] = (
    0.45 * rgb[visible_mask].astype(np.float32) + 0.55 * green
).astype(np.uint8)

fig_rgb = go.Figure(go.Image(z=rgb_overlay))
fig_rgb.update_layout(
    title=f"场景 {scene_dir.name}，图像 {image_id:06d}，gt_id={gt_id} 的可见掩码",
    width=800,
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_rgb.show()

# 掩码以外或没有有效深度的像素设为 NaN，在热力图中留空。
depth_for_plot = np.where(object_depth_mask, depth_mm, np.nan)
fig_depth = go.Figure(go.Heatmap(
    z=depth_for_plot,
    colorscale="Viridis",
    colorbar=dict(title="深度 (mm)"),
    hovertemplate="u=%{x}, v=%{y}<br>深度=%{z:.1f} mm<extra></extra>",
))
fig_depth.update_xaxes(title="u（像素）")
fig_depth.update_yaxes(title="v（像素）", autorange="reversed", scaleanchor="x")
fig_depth.update_layout(
    title="当前实例的有效深度",
    width=800,
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_depth.show()

## 3. 将可见深度反投影为相机坐标点云

图像数组按 `[v, u]` 访问：`v` 是行号，`u` 是列号。对每个保留像素，使用相机内参进行反投影：

$$
\mathbf{p}_C = Z_C K^{-1}\begin{bmatrix}u\\v\\1\end{bmatrix}
$$

`K` 来自同一帧的 `scene_camera.json`，$Z_C$ 来自该像素的深度值。得到的三维点位于相机坐标系 C，单位为毫米。

In [ ]:
# np.nonzero 返回行索引 v 和列索引 u。
v_coords, u_coords = np.nonzero(object_depth_mask)
z_values = depth_mm[v_coords, u_coords].astype(np.float64)

# 将像素写成齐次坐标 [u, v, 1]，再乘 K 的逆得到相机射线方向。
pixels_h = np.vstack((u_coords, v_coords, np.ones_like(u_coords))).astype(np.float64)
rays_c = np.linalg.solve(K, pixels_h)

# 每条相机射线乘以该像素的深度，恢复相机坐标中的三维点。
observed_points_c = (rays_c * z_values[None, :]).T

print(f"观测点云形状: {observed_points_c.shape}（点数 × XYZ）")
print("前 5 个相机坐标点 (mm):")
print(np.round(observed_points_c[:5], 2))

## 4. 将对应 CAD 点云变换到同一相机坐标系

CAD 网格点最初位于物体坐标系 O。当前实例的 `obj_id` 用于选择 CAD 文件，真值位姿 $T_{CO}$ 用于执行坐标变换：

$$
\mathbf{p}^{CAD}_C = R_{CO}\mathbf{p}_O + \mathbf{t}_{CO}
$$

变换后，深度观测点和 CAD 点均位于相机坐标系 C，单位为毫米，可在同一 3D 图中比较。

In [ ]:
model_path = MODELS_DIR / f"obj_{obj_id:06d}.ply"
mesh = o3d.io.read_triangle_mesh(str(model_path))
if mesh.is_empty():
    raise ValueError(f"CAD 网格读取失败或为空: {model_path}")

# 从 CAD 表面均匀采样点；采样点仍处于物体坐标系 O。
CAD_POINT_COUNT = 6000
cad_cloud = mesh.sample_points_uniformly(number_of_points=CAD_POINT_COUNT)
cad_points_o = np.asarray(cad_cloud.points)

# 使用当前实例的 BOP 真值位姿，把 CAD 点从 O 系变换到相机坐标系 C。
cad_points_c = (R_co @ cad_points_o.T).T + t_co

print(f"obj_id {obj_id} 对应的 CAD 文件: {model_path}")
print(f"CAD 表面采样点数: {len(cad_points_c)}")
print(f"观测点和 CAD 点的坐标系: 相机坐标系 C")
print("两组点的单位: 毫米")

## 5. 同屏比较两组点云

蓝色点来自深度图和实例可见掩码，覆盖当前相机可见且具有有效深度的表面。橙色点来自完整 CAD 网格，并按真值位姿变换到相机坐标系，因此包含被遮挡的背面。

深度噪声、孔洞、边缘误差以及 CAD 与实物的细小差异会导致两组点云存在偏差。本单元使用真值位姿进行对照，不输出估计位姿。

In [ ]:
fig_cloud = go.Figure()
fig_cloud.add_trace(go.Scatter3d(
    x=observed_points_c[:, 0],
    y=observed_points_c[:, 1],
    z=observed_points_c[:, 2],
    mode="markers",
    marker=dict(size=2, color="royalblue", opacity=0.9),
    name=f"深度观测可见点 ({len(observed_points_c)})",
))
fig_cloud.add_trace(go.Scatter3d(
    x=cad_points_c[:, 0],
    y=cad_points_c[:, 1],
    z=cad_points_c[:, 2],
    mode="markers",
    marker=dict(size=2, color="darkorange", opacity=0.35),
    name=f"真值位姿下的完整 CAD ({len(cad_points_c)})",
))
fig_cloud.update_layout(
    title=f"场景 {scene_dir.name}，图像 {image_id:06d}，obj_id={obj_id}：相机坐标系点云对照",
    scene=dict(
        xaxis_title="X_C (mm)",
        yaxis_title="Y_C (mm)",
        zaxis_title="Z_C (mm)",
        aspectmode="data",
    ),
    width=1000,
    height=750,
    margin=dict(l=10, r=10, t=60, b=10),
    legend=dict(itemsizing="constant"),
)
fig_cloud.show()

## 本阶段小结

本阶段完成了以下数据与几何处理：

- `mask_visib` 标出当前实例在图像中的可见区域；
- 深度值乘以 `depth_scale` 后转换为毫米单位的相机深度；
- 相机内参 $K$ 将有效深度像素反投影为相机坐标系点云；
- BOP 真值位姿将 CAD 点变换到相机坐标系；
- 深度点云表示可见且有有效深度的部分，CAD 点云表示完整表面。

### 自测问题

1. 为什么反投影时要同时检查掩码和深度是否有效？
2. `depth_scale` 作用于哪个数值？为什么不能直接把原始深度当作毫米？
3. 为什么两组点云需要位于同一坐标系才能直接对照？
4. 为什么深度观测点云通常只覆盖 CAD 模型的一部分？

完成本阶段后，可进入阶段三，学习粗配准与 ICP 位姿细化。